# Dictionaries, Sets, and Reconciliation


```{contents}
:local:
:depth: 2
```


```{index} A Set Answers Membership; a Count Answers Multiplicity
```



```{index} A Set Answers Membership; a Count Answers Multiplicity
```

## A Set Answers Membership; a Count Answers Multiplicity



Two departments submit product IDs for reconciliation. A set represents which IDs occur, regardless of how many times they were entered. A dictionary of counts represents how often each ID occurs. Choosing between them changes the information retained, not merely the syntax of the container.

Let the warehouse IDs be {A, B, C} and the sales IDs be {B, C, D}. Intersection gives IDs recognized by both: {B, C}. Warehouse minus sales gives {A}; sales minus warehouse gives {D}. The direction of difference matters. A union gives all recognized IDs {A, B, C, D}; it does not establish that either department's quantities agree.

| Question | Operation | Result for this example |
| --- | --- | --- |
| Which IDs are shared? | Intersection | B, C |
| Which warehouse IDs are absent from sales? | Warehouse minus sales | A |
| Which sales IDs are absent from warehouse? | Sales minus warehouse | D |
| Which IDs appear anywhere? | Union | A, B, C, D |

Now suppose sales submits B twice. A set still contains one B. That is correct for unique membership and wrong for a task that must detect repeated submissions. A frequency dictionary can increment B's count to two. State whether duplicates are errors, repeated events, or information to discard before building either representation.

For related explanation, see the further reading below.[^development-reading]


```{index} Choose Equality Before Comparing the Groups
```



```{index} Choose Equality Before Comparing the Groups
```

## Choose Equality Before Comparing the Groups



If product IDs are case-insensitive under the business contract, B and b must use compatible equality in both collections. Normalizing only one side can produce false missing-ID reports. For hashed keys, equal keys must have equal hash codes under the selected policy. A good hash distribution improves performance but cannot repair inconsistent equality.

Do not mutate fields that determine a key's hash/equality while that key is stored. The entry may remain physically present in a bucket chosen by its old meaning while a new lookup examines another location. Immutable key data or remove/change/reinsert workflows make that boundary explicit.

Set-operation APIs also have mutation contracts. Some operations modify the receiving set; query-based forms may produce a sequence instead. If the original warehouse membership must remain available, work on a copy or choose a nonmutating approach deliberately. Assigning another variable to the same set object does not create independent storage.

Guided exercise: add B twice and lowercase a once to the sales input. Produce two answers: a unique-ID reconciliation under a stated case policy and a repeated-ID report. Explain which information a set has discarded and why the frequency report cannot be reconstructed from that set alone.


Keyed access and membership answer different questions. A dictionary maps an ID to a record. A set records whether an ID belongs to a group. Business reconciliation often needs set operations as well as individual lookups.

Use the same equality rule for all collections that participate in one calculation. The examples use ordinal case-insensitive equality for submitted symbolic IDs and preserve original sets when a report needs several answers.


```{index} Sets and Frequency Counts
```

## Sets and Frequency Counts




```{index} HashSet; set; frequency count
```

A set stores keys without associated record values. Use `HashSet<T>.Add` to detect a first occurrence, or a dictionary to count occurrences. Give both the same comparer when they must agree about what counts as equal.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

string[] statuses = { "Open", "open", "Closed", "OPEN", "closed" };
var seen = new HashSet<string>(StringComparer.OrdinalIgnoreCase);
var counts = new Dictionary<string, int>(StringComparer.OrdinalIgnoreCase);
foreach (string status in statuses)
{
    seen.Add(status);
    counts.TryGetValue(status, out int count);
    counts[status] = count + 1;
}
Console.WriteLine($"distinct statuses={seen.Count}");
foreach (string status in new[] { "open", "closed" })
    Console.WriteLine($"{status}: {counts[status]}");

distinct statuses=2
open: 3
closed: 2


The set has two distinct statuses, and the counts are open three and closed two. The example prints keys in an explicit order; a hash collection's enumeration order is not a sorting contract. Sort keys separately when a report requires an order. Neither a set nor a dictionary supplies the priority-selection operation of a heap.

**Common mistake.** Giving the set and the dictionary different comparers. With a case-sensitive dictionary, `Open` and `open` become separate keys while the case-insensitive set counts one status, and the two reports disagree about the same data.


```{index} Code Completion: Count Occurrences
```

### Code Completion: Count Occurrences

Complete the loop so that `counts` records how many orders each customer placed. Use the pattern that handles the first occurrence and later occurrences without two lookups: read the current count with `TryGetValue`, treating absence as zero, then store the incremented value. The expected output is `C-104: 3`, `C-117: 1`, `C-121: 2`, printed in the order given by the sorted keys.

In [ ]:
using System;
using System.Collections.Generic;

string[] orders = { "C-104", "C-121", "C-104", "C-117", "C-121", "C-104" };
var counts = new Dictionary<string, int>(StringComparer.Ordinal);

foreach (string customer in orders)
{
    // Your code starts here.

    // Your code ends here.
}

var keys = new List<string>(counts.Keys);
keys.Sort(StringComparer.Ordinal);
foreach (string key in keys) Console.WriteLine($"{key}: {counts[key]}");

In [ ]:
// Solution
using System;
using System.Collections.Generic;

string[] orders = { "C-104", "C-121", "C-104", "C-117", "C-121", "C-104" };
var counts = new Dictionary<string, int>(StringComparer.Ordinal);

foreach (string customer in orders)
{
    counts.TryGetValue(customer, out int current);   // current is 0 when the key is absent
    counts[customer] = current + 1;                  // indexer assignment inserts or replaces
}

var keys = new List<string>(counts.Keys);
keys.Sort(StringComparer.Ordinal);
foreach (string key in keys) Console.WriteLine($"{key}: {counts[key]}");

C-104: 3
C-117: 1
C-121: 2


`TryGetValue` leaves `current` at its default of 0 when the key is absent, so the first occurrence and every later one run the same two lines. The common alternative, `if (counts.ContainsKey(c)) counts[c]++; else counts[c] = 1;`, is correct but hashes the key twice on the frequent path. Writing `counts[customer]++` alone throws `KeyNotFoundException` on the first occurrence, because the indexer's getter does not create missing keys.

```{index} Union, Intersection, and Difference
```

## Union, Intersection, and Difference




```{index} Union, Intersection, and Difference
```


Let registered contain everyone registered for a workshop and attended contain everyone who checked in. Their intersection contains registered attendees. Registered minus attended identifies no-shows. Attended minus registered identifies walk-ins. Their union contains everyone appearing in either source.

HashSet operations such as IntersectWith and ExceptWith mutate the receiving set. Construct a copy first when you need to retain both source sets. Enumeration is not a sorted-report contract: copy to a list and sort explicitly for predictable output.


In [1]:
using System;
using System.Collections.Generic;

var registered=new HashSet<string>(new[]{"A","B","C"},StringComparer.OrdinalIgnoreCase);
var attended=new HashSet<string>(new[]{"b","C","D"},StringComparer.OrdinalIgnoreCase);
var both=new HashSet<string>(registered,StringComparer.OrdinalIgnoreCase);both.IntersectWith(attended);
var absent=new HashSet<string>(registered,StringComparer.OrdinalIgnoreCase);absent.ExceptWith(attended);
var all=new HashSet<string>(registered,StringComparer.OrdinalIgnoreCase);all.UnionWith(attended);
foreach(var group in new[]{both,absent,all}){var report=new List<string>(group);report.Sort(StringComparer.OrdinalIgnoreCase);Console.WriteLine(string.Join(",",report));}
Console.WriteLine($"Sources: {registered.Count}, {attended.Count}");


B,C
A
A,B,C,D
Sources: 3, 3


The output is `B,C`, `A`, `A,B,C,D`, and `Sources: 3, 3`. The lowercase `b` in `attended` matched `B` because both sets use the case-insensitive comparer. The two source counts are still 3 because each operation ran on a copy; call `registered.IntersectWith` directly and the second report would be computed from a set that had already lost A.

**Common mistake.** Calling `registered.IntersectWith(attended)` directly. The method rewrites `registered` in place, so the later no-show difference is computed from an already shrunken set; copy first, as the cell does, when more than one answer is needed.


```{index} Lookup and Missing Values
```

## Lookup and Missing Values




```{index} Lookup and Missing Values
```


TryGetValue separates an absent key from a stored default value. Do not use a balance of zero, an empty owner name, or a null-like sentinel as an implicit membership test unless the contract explicitly excludes those values.

Choose Add when duplicate keys are errors; use indexer assignment when replacement is intended. If imports may repeat customer IDs, state whether a repeated row updates, accumulates, or rejects. Hashing provides access; it does not choose the business policy.


In [1]:
using System;
using System.Collections.Generic;

var units=new Dictionary<string,int>(StringComparer.OrdinalIgnoreCase);
units["P100"]=0;
foreach(string id in new[]{"p100","missing"})
    Console.WriteLine(units.TryGetValue(id,out int count)?$"{id}: stored {count}":$"{id}: missing");


p100: stored 0
missing: missing


The output is `p100: stored 0` and `missing: missing`. The lookup for `p100` found the key stored as `P100` because the dictionary's comparer ignores case, and it reported the stored zero as a value rather than as absence. A test such as `units[id] == 0` could not tell these two lines apart.

**Common mistake.** Reading `units["missing"]` to see whether a key exists. The indexer throws `KeyNotFoundException` on absence, and testing `units[id] == 0` cannot distinguish a stored zero from a missing key; `TryGetValue` answers both questions at once.


```{index} Keys Must Keep Their Meaning
```

## Keys Must Keep Their Meaning




```{index} Keys Must Keep Their Meaning
```


Do not change fields that determine a key's equality or hash while that key is stored. The key can become inconsistent with its original bucket. Immutable string IDs avoid that mutation hazard. A custom business object may instead need a comparer based on an immutable ID, as introduced in Chapter 13.

A hash collection does not provide sorted order, range queries, or next-priority removal. For those requirements choose another structure, or maintain multiple indexes with a clear consistency policy. The next section coordinates dictionary membership with the priority queue from Chapter 18.

The hazard is concrete enough to demonstrate. A key object whose `Equals` and `GetHashCode` read a mutable field is stored in a set, the field is changed, and the set can no longer find the object even though it is still inside: the lookup computes the new hash, goes to a different bucket, and reports absence, while `Count` still says 1 and enumeration still yields the object. The example below shows exactly that sequence.

In [ ]:
using System;
using System.Collections.Generic;

var tag = new MutableTag("P100");
var catalog = new HashSet<MutableTag> { tag };
Console.WriteLine($"before mutation: contains={catalog.Contains(tag)}, count={catalog.Count}");

tag.Code = "P200";                                   // changes Equals and GetHashCode of a stored key
Console.WriteLine($"after mutation:  contains={catalog.Contains(tag)}, count={catalog.Count}");
Console.WriteLine($"enumeration still yields: {string.Join(", ", catalog)}");
Console.WriteLine($"can it be removed? {catalog.Remove(tag)}");

class MutableTag
{
    public string Code { get; set; }
    public MutableTag(string code) => Code = code;
    public override bool Equals(object? obj) => obj is MutableTag other && other.Code == Code;
    public override int GetHashCode() => Code.GetHashCode();
    public override string ToString() => Code;
}

before mutation: contains=True, count=1
after mutation:  contains=False, count=1
enumeration still yields: P200
can it be removed? False


Before the mutation the set finds its own key. After `Code` changes, `Contains` returns `False` and `Remove` fails, yet `Count` is still 1 and the object is still enumerated: it sits in the bucket chosen by the old hash code, and every lookup now searches the bucket for the new one. The entry is unreachable by key and cannot be deleted by key, which in a long-running service is a memory leak and a correctness bug at once. The remedies are the ones named above: make the key type immutable, as `record` types and strings are, or remove the entry, change the object, and add it back.

```{index} Concept Check: Which Operation Answers the Question?
```

### Concept Check: Which Operation Answers the Question?

A warehouse list holds product IDs {A, B, C, E} and a sales list holds {B, C, D, E, E}. For each question choose the operation and give the result, then open the answer.

1. Which IDs did both departments report?
2. Which IDs did the warehouse report that sales never did?
3. Which IDs appear in either report?
4. Did sales report any ID more than once, and could a set tell you?

```{dropdown} Answer
1. Intersection: {B, C, E}.
2. Warehouse minus sales: {A}. The reverse difference, sales minus warehouse, is {D}, a different question.
3. Union: {A, B, C, D, E}.
4. Yes, E twice, and a set cannot tell you: it stores E once. Only a frequency dictionary, or a count of the input against the set's size, reveals the duplicate.
```

```{index} Practice: Report Missing IDs
```

## Practice: Report Missing IDs




```{index} Practice: Report Missing IDs
```


Produce planned minus actual without modifying the planned set.


In [ ]:
using System;
using System.Collections.Generic;

var planned=new HashSet<string>(new[]{"A","B","C"},StringComparer.Ordinal);
var actual=new HashSet<string>(new[]{"B","D"},StringComparer.Ordinal);
var missing=new HashSet<string>(planned,StringComparer.Ordinal);// Your code starts here.
/* TODO: keep only IDs absent from actual. */
// Your code ends here.
var report=new List<string>(missing);report.Sort(StringComparer.Ordinal);
Console.WriteLine(string.Join(",",report));


In [1]:
using System;
using System.Collections.Generic;

var planned=new HashSet<string>(new[]{"A","B","C"},StringComparer.Ordinal);
var actual=new HashSet<string>(new[]{"B","D"},StringComparer.Ordinal);
var missing=new HashSet<string>(planned,StringComparer.Ordinal);missing.ExceptWith(actual);
var report=new List<string>(missing);report.Sort(StringComparer.Ordinal);
Console.WriteLine(string.Join(",",report));


A,C


The output is `A,C`: the planned IDs that never appeared in `actual`. `planned` still holds three IDs afterward, because `ExceptWith` ran on the copy named `missing`. Reversing the operands, actual minus planned, would report `D`, a different business question.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. `HashSet<string>.Add` returned `false`. What happened to the set?

```{dropdown} Answer
Nothing. `Add` reports whether the element was new; a `false` result means an equal element was already present, and the set is unchanged. That return value is the idiomatic way to detect a first occurrence in one operation.
```

2. A dictionary uses `StringComparer.OrdinalIgnoreCase` and the program later looks up keys with `StringComparer.Ordinal` semantics in a second set. Why can the two disagree?

```{dropdown} Answer
Because equality defines membership. `Open` and `OPEN` are one key in the first collection and two in the second, so counts, intersections, and missing-ID reports computed across the pair are inconsistent. Every collection in one calculation must share the equality policy.
```

3. Why is `TryGetValue` preferred over `ContainsKey` followed by the indexer?

```{dropdown} Answer
It answers presence and retrieves the value in one hash lookup, and it distinguishes a stored default value from an absent key through its Boolean result. The two-call form hashes the key twice and invites the mistake of using the value as the presence test.
```

4. `IntersectWith` changed the set it was called on. Which other `HashSet<T>` methods behave the same way, and how do you keep the original?

```{dropdown} Answer
`UnionWith`, `ExceptWith`, and `SymmetricExceptWith` all rewrite the receiving set. Construct a copy with the set constructor, passing the same comparer, and call the operation on the copy; the `Overlaps` and `IsSubsetOf` family are queries and do not mutate.
```

```{rubric} Footnotes
```


[^1902_dictionaries_sets-1]: Set difference is directional. Registered minus attended and attended minus registered answer different business questions.


[^development-reading]: Further reading: [OpenDSA, Introduction to Hashing](https://opendsax.cs.vt.edu/OpenDSA/Books/Everything/html/HashIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
